In [1]:
%pip install -q requests python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [ ]:
import os
import requests
from dotenv import load_dotenv


# --------------------------------------------------
# LOAD OUR SECRET API KEYS
# --------------------------------------------------

# load_dotenv() reads the .env file
# and makes our API keys available to Python.
#
# Example .env:
#
# GEMINI_API_KEY=your_key
# GROQ_API_KEY=your_key
# OPENROUTER_API_KEY=your_key

load_dotenv()


# --------------------------------------------------
# CREATE OUR AI FUNCTION
# --------------------------------------------------

# "def" means DEFINE.
#
# We are creating our own function called "ask".
#
# Think of a function like a machine:
#
#     INPUT  →  MACHINE  →  OUTPUT
#
# provider = which AI should we use?
# prompt   = what question should we ask?
#
# Example:
#
# ask("gemini", "What is Python?")

def ask(provider, prompt):

    """
    Our AI waiter.

    provider = the AI we want to talk to
    prompt   = the question we want to ask

    Example:

        ask("gemini", "Explain APIs")
    """


    # --------------------------------------------------
    # 1. CHOOSE WHICH AI WE WANT TO USE
    # --------------------------------------------------

    # "if" means:
    #
    # "If this condition is true, do this."

    if provider == "gemini":

        # The Gemini API address
        url = "https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent"

        # Our API key is like our ticket.
        #
        # os.getenv() means:
        # "Python, go and get this value from .env"

        headers = {
            "x-goog-api-key": os.getenv("GEMINI_API_KEY")
        }

        # This is the question we are sending to Gemini.
        #
        # "prompt" contains whatever question
        # the user gave us.

        body = {
            "contents": [
                {
                    "parts": [
                        {"text": prompt}
                    ]
                }
            ]
        }


    # --------------------------------------------------
    # GROQ
    # --------------------------------------------------

    # "elif" means:
    #
    # "ELSE IF"
    #
    # In simple English:
    #
    # "If the previous condition was NOT true,
    # check this one."

    elif provider == "groq":

        # Groq API address
        url = "https://api.groq.com/openai/v1/chat/completions"

        # Our Groq API key
        headers = {
            "Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}",
            "Content-Type": "application/json"
        }

        # The question we are sending
        body = {
            "model": "openai/gpt-oss-20b",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }


    # --------------------------------------------------
    # OPENROUTER
    # --------------------------------------------------

    # If it was NOT Gemini
    # AND it was NOT Groq,
    # check if it is OpenRouter.

    elif provider == "openrouter":

        # OpenRouter API address
        url = "https://openrouter.ai/api/v1/chat/completions"

        # Our OpenRouter API key
        headers = {
            "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
            "Content-Type": "application/json"
        }

        # openrouter/free means:
        #
        # "Give me an available free model."
        #
        # OpenRouter chooses an available free model
        # instead of us choosing one manually.

        body = {
            "model": "openrouter/free",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }


    # --------------------------------------------------
    # OLLAMA
    # --------------------------------------------------

    # If it was NOT Gemini,
    # NOT Groq,
    # NOT OpenRouter,
    # check if it is Ollama.

    elif provider == "ollama":

        # Ollama runs on our own computer.
        #
        # That's why we use localhost.
        #
        # localhost = "this computer"

        url = "http://localhost:11434/api/chat"

        # Ollama does not need an API key
        # because it is running locally.

        headers = {
            "Content-Type": "application/json"
        }

        # The question we are sending to Ollama

        body = {
            "model": "llama3.2",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "stream": False
        }


    # --------------------------------------------------
    # UNKNOWN AI
    # --------------------------------------------------

    # "else" means:
    #
    # "None of the above worked."
    #
    # For example:
    #
    # ask("facebook", "Explain APIs")
    #
    # We don't have Facebook in our function,
    # so Python will come here.

    else:
        raise ValueError("Unknown provider")


    # --------------------------------------------------
    # 2. SEND THE QUESTION TO THE AI
    # --------------------------------------------------

    # requests.post() sends our question
    # to the AI server.
    #
    # Think of POST as:
    #
    # "Here is my question. Please process it."

    response = requests.post(
        url,
        headers=headers,
        json=body
    )


    # --------------------------------------------------
    # 3. CHECK FOR ERRORS
    # --------------------------------------------------

    # If the server says something went wrong,
    # this will show the error.

    response.raise_for_status()


    # --------------------------------------------------
    # 4. READ THE AI'S RESPONSE
    # --------------------------------------------------

    # The AI sends its answer back as JSON.
    #
    # .json() converts that response into
    # something Python can understand.

    data = response.json()


    # --------------------------------------------------
    # 5. GET THE ACTUAL ANSWER
    # --------------------------------------------------

    # Gemini puts its answer in a different
    # place inside the JSON response.

    if provider == "gemini":

        return data["candidates"][0]["content"]["parts"][0]["text"]


    # Ollama also has a different response format.

    elif provider == "ollama":

        return data["message"]["content"]


    # Groq and OpenRouter use this format.

    else:

        return data["choices"][0]["message"]["content"]

In [15]:
ask("gemini", "Explain APIs")

HTTPError: 400 Client Error: Bad Request for url: https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent

In [ ]:
ask("groq", "Explain APIs")

HTTPError: 401 Client Error: Unauthorized for url: https://api.groq.com/openai/v1/chat/completions

In [ ]:
ask("ollama", "Explain APIs")